# 双鞭毛虫：100 Hz 双侧神经网
顺序运行各单元。代码取自 GitHub；优先保存到 Google Drive，挂载失败时保存到 Colab 本地并下载压缩包；最后生成外部 Web 演示链接。

In [ ]:
!if [ -d /content/Parrot/.git ]; then git -C /content/Parrot pull --ff-only; else git clone -q https://github.com/DoctorAwe/Parrot.git /content/Parrot; fi
%cd /content/Parrot
!pip -q install numpy gradio plotly

## 1. 选择结果目录并检查 100 tick/s 模拟
优先挂载 Google Drive。若挂载失败则继续运行，结果暂存于 `/content/proto_microbe_v1`；Colab 运行时结束前需下载。

In [ ]:
from pathlib import Path
from google.colab import drive
drive_ok = False
try:
    drive.mount('/content/drive', timeout_ms=30000)
    drive_ok = Path('/content/drive/MyDrive').is_dir()
except (ValueError, OSError, TimeoutError) as exc:
    print(f'Drive 挂载失败：{exc}。改用 Colab 本地目录，结束前下载结果。')
out = (Path('/content/drive/MyDrive/Parrot/proto_microbe_v1') if drive_ok
       else Path('/content/proto_microbe_v1'))
out.mkdir(parents=True, exist_ok=True)
print('结果目录:', out)
from proto_microbe import ProtoConfig, BilateralNerveNet, rollout, evaluate, evolve, replay_figure
cfg = ProtoConfig(steps=3000, dt=0.01, neuron_count=128, fan_in=12)
assert cfg.dt == 0.01
net = BilateralNerveNet(cfg, seed=404)
print(f'{net.n} neurons, {net.synapses} synapses, {1/cfg.dt:.0f} ticks/s')
print('Reflex smoke test:', rollout(net, net.reflex_genome(), 3000))

## 2. 训练
固定神经连接拓扑，扰动突触权重。在训练世界排序，在独立验证世界决定是否保留。约需数分钟，取决于 Colab 运行时。每代自动保存检查点。

In [ ]:
best, history = evolve(net, generations=12, population=16,
                       train_seeds=range(1000, 1006),
                       validation_seeds=range(2000, 2006), out=out)
print('Checkpoint:', out)

## 3. 独立测试
测试种子与训练、验证种子均不重叠。观察是否真正超过起点反射。

In [ ]:
import json, numpy as np
test_seeds = range(3000, 3020)
rng = np.random.default_rng(99)
results = {
    'reflex': evaluate(net, net.reflex_genome(), test_seeds),
    'trained': evaluate(net, best, test_seeds),
    'random': evaluate(net, net.random_genome(rng), test_seeds),
}
print(json.dumps(results, indent=2))
(out / 'test_results.json').write_text(json.dumps(results, indent=2), encoding='utf-8')

## 4. 轨迹和 Web 外链
单次轨迹仅供可视化，不进入模型。Gradio 启动后打印外部链接；点击会在新标签页打开。Colab 运行时需保持运行。

In [ ]:
score, replay = rollout(net, best, 3000, record=True)
(out / 'replay_seed3000.json').write_text(json.dumps(replay), encoding='utf-8')
fig = replay_figure(replay, every=30)
fig.show()

In [ ]:
if not drive_ok:
    import shutil
    from google.colab import files
    archive = shutil.make_archive('/content/proto_microbe_v1_results', 'zip', root_dir=out)
    print('下载结果压缩包:', archive)
    files.download(archive)
else:
    print('结果已保存在 Google Drive:', out)

In [ ]:
from proto_web import create_demo
from IPython.display import HTML, display
demo = create_demo(net, best)
demo.launch(share=True, inline=False, show_error=True)
if demo.share_url:
    display(HTML(f'<p><a href="{demo.share_url}" target="_blank" rel="noopener">在新标签页打开双鞭毛虫 Web 演示</a></p>'))
    print('External demo:', demo.share_url)
else:
    print('未获得外链；查看上方 Gradio 启动日志。')